# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant Dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

In [ ]:
# List available record sets and their @id fields
print('Available record sets (@id):')
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    print(f"  {rs_id}")

# For each record set, show fields and their @id values
for rs_id in record_sets:
    recset = dataset.record_sets[rs_id]
    print(f"\nRecord Set: {rs_id}")
    print(f"  Name: {recset.name}")
    print(f"  Description: {recset.description if hasattr(recset, 'description') else 'No description'}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - @id: {field.id}")

## 3. Data Extraction
Load data from specified record sets into pandas DataFrames for analysis. All record set and field references use their `@id` fields.

In [ ]:
# Extract records from all available record sets into DataFrames, keyed by @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns of each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nRecord set @id: {rs_id}")
    print(f"Columns: {list(df.columns)}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All references use the `@id` fields from the previous section.

In [ ]:
###
# For demonstration, automatically select the first record set that contains at least one numeric column
import numpy as np

# Utility to find first numeric field in the record sets
target_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    if not df.empty:
        # Try to detect first numeric column
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        object_cols = df.select_dtypes(include=['object']).columns.tolist()
        if numeric_cols:
            target_record_set_id = rs_id
            numeric_field_id = numeric_cols[0]
            # For group field, pick the first non-numeric column
            if object_cols:
                group_field_id = object_cols[0]
            break

if target_record_set_id is None or numeric_field_id is None:
    print('No suitable numeric field found in any record set.')
else:
    print(f"Analysing record set: {target_record_set_id}, numeric field: {numeric_field_id}")
    
    df = dataframes[target_record_set_id]

    # Remove NA values for reliable filtering
    df = df.dropna(subset=[numeric_field_id])

    # Set a threshold (use the 75th percentile to filter top quartile)
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in '{target_record_set_id}' with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalizing the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate if a group field is available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped statistics by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields in the dataset.

Below, we visualize the distribution of the selected numeric field and its relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=filtered_df, x=numeric_field_id, kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, explore, and visualize a multilayered dataset described by a Croissant schema. The exploration included overview and extraction of record sets using their `@id`, selection and normalization of numeric fields, and grouped aggregations by categorical fields, all referenced by `@id` throughout. This workflow offers a robust, reproducible foundation for in-depth analysis of FAIR datasets using Croissant standards.